# Güncel Acemi – Stockfish güç testi: 500 ms, 100 + 100 maç

Acemi'nin güncel main sürümü **38b0722** sabitlenir; TT ve paketli değerlendirme
açıktır. **Stockfish UCI_Elo=1320 karşısında tek toplu 100 maç**, ardından
**UCI_Elo=1500 karşısında tek toplu 100 maç** oynanır. Stockfish'in standart
alt sınırı 1320 olduğundan 1300 civarı hedefi 1320 ile sınanır. UCI_LimitStrength
iki grupta da true; her iki motora hamle başına **500 ms** verilir.

İki seviyede aynı 50 dengeli açılış kullanılır: her açılışta bir beyaz/bir siyah,
seviye başına 50/50 renk dağılımı. Kitap ve SPRT kapalı; Hash 16 MB, tek thread,
concurrency 2. Oyunlar 10'luk bölümlere ayrılmaz. Arama ve beraberlik politikası
bu test için değiştirilmez. Toplam **200 maç**; bütün maçlar yalnız Kaggle'da.

`codex/stockfish-500ms` dalından içe alın, Internet ve Contents yazma yetkili
GITHUB_TOKEN secret'ını açın; **Run All** çalıştırın. Hazırlıkta motor sürümü,
binary özeti ve iki Elo ayarının desteklendiği doğrulanır. Sonuçlar her 100 maç
sonunda GitHub'a gönderilir. Son hücre yalnız push tekrarlar.

Stockfish UCI_Elo bir güç sınırlama hedefidir; sonuçlardan FIDE/Chess.com/Lichess
puanı doğrudan çıkarılmaz. [Resmî Stockfish seçenekleri](https://official-stockfish.github.io/docs/stockfish-wiki/UCI-Protocol-and-Stockfish-Commands.html).


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/stockfish-500ms"
WORKSPACE = "/kaggle/working/acemi-stockfish-strength"
SESSION_ID = "main-stockfish-500ms-100each-v1"
ENGINE_COMMIT = "38b0722b084552d8e94eb62e3558d447b641969c"
ENGINE_SOURCE = "/kaggle/working/acemi-strength-main"
RETRY_INCOMPLETE = False


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("This strength test runs only in Kaggle.")

def checked(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {args[0]}\n{(result.stdout + result.stderr)[-4000:]}")
    return result.stdout.strip()

workspace = Path(WORKSPACE)
if not workspace.exists():
    checked(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)])
else:
    checked(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local = subprocess.run(["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"])
    if local.returncode == 0:
        checked(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        checked(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH, "--track", f"origin/{GITHUB_BRANCH}"])
    checked(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
if not (workspace / "tools/stockfish_strength.py").exists():
    raise RuntimeError("Import the latest notebook from codex/stockfish-500ms.")
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"], cwd=workspace)
stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    checked(["apt-get", "update", "-qq"])
    checked(["apt-get", "install", "-y", "-qq", "stockfish"])
    stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    raise RuntimeError("Stockfish executable was not found after installation.")
engine_source = Path(ENGINE_SOURCE)
if not engine_source.exists():
    checked(["git", "clone", "--depth", "1", "--no-checkout", GITHUB_REPO_URL, str(engine_source)])
checked(["git", "-C", str(engine_source), "fetch", "--depth", "1", "origin", ENGINE_COMMIT])
checked(["git", "-C", str(engine_source), "checkout", "--detach", ENGINE_COMMIT])
if checked(["git", "-C", str(engine_source), "rev-parse", "HEAD"]) != ENGINE_COMMIT:
    raise RuntimeError("Pinned main source mismatch")
session = workspace / "data" / "strength" / SESSION_ID
print("Pinned main:", ENGINE_COMMIT, "Stockfish:", stockfish_bin)
print("500 ms for BOTH engines; SF 1320: 100 games, SF 1500: 100 games; total: 200.")


In [ ]:
# Publish records only. The token is never written to a file or a remote URL.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GITHUB_TOKEN under Kaggle Add-ons / Secrets.")
    auth = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": auth, "GIT_TERMINAL_PROMPT": "0"})
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stdout + result.stderr).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
            raise RuntimeError(f"Git {args[0]} failed: {details[-2000:]}")
        return result
    if git("branch", "--show-current").stdout.strip() != GITHUB_BRANCH:
        raise RuntimeError("Wrong results branch; rerun repository setup.")
    git("config", "user.name", "Acemi Kaggle A-B")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    paths = [str(session.relative_to(workspace))]
    paths += [p for p in ("data/runs", "data/index.csv", "data/positions") if (workspace / p).exists()]
    git("add", "--", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()):
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Fresh processes: no stale tools or engine imports from an earlier Kaggle notebook.
print(checked([sys.executable, "-m", "unittest", "tests.test_stockfish_strength", "-v"], cwd=workspace))
print(checked([sys.executable, "-u", "-m", "tools.stockfish_strength", "prepare", "--id", SESSION_ID,
               "--source", str(engine_source), "--stockfish", stockfish_bin], cwd=workspace))
plan = json.loads((session / "protocol.json").read_text(encoding="utf-8"))
print("Stockfish:", plan["stockfish"]["name"], "supported UCI_Elo:", plan["stockfish"]["elo_option"])
print("Engine source:", plan["source"], "groups:", plan["groups"])
publish_results(f"data: prepare current-main Stockfish strength test ({SESSION_ID})")


In [ ]:
def run_level(elo):
    command = [sys.executable, "-u", "-m", "tools.stockfish_strength", "run", "--session", str(session), "--elo", str(elo)]
    if RETRY_INCOMPLETE:
        command.append("--retry-incomplete")
    result = subprocess.run(command, cwd=workspace)
    publish_results(f"data: current-main vs Stockfish {elo}, 500ms 100 games ({SESSION_ID})")
    if result.returncode:
        raise RuntimeError(f"Stockfish {elo} run stopped; logs and raw results are retained. Inspect before retrying.")
    print((session / "SUMMARY.md").read_text(encoding="utf-8"))

# ONE 100-game call. Completed groups are verified and skipped when rerun.
run_level(1320)


In [ ]:
# A second ONE 100-game call against the other Stockfish level.
run_level(1500)


In [ ]:
# Safe push retry only; this cell starts no games or engines.
publish_results(f"data: finalize current-main Stockfish strength results ({SESSION_ID})")
if (session / "SUMMARY.md").exists():
    print((session / "SUMMARY.md").read_text(encoding="utf-8"))
print("Compare each opponent's result separately; the target UCI_Elo is not a human rating.")
